In [ ]:
import requests
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm
import time

In [2]:
EMAIL = "aashray.godani007@gmail.com"

DATA_DIR = "../data/raw_papers"
os.makedirs(DATA_DIR, exist_ok=True)

In [3]:
SEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"

params = {
    "db": "pmc",
    "term": "diabetes OR cancer OR cardiovascular disease",
    "retmax": 300,
    "retmode": "json",
    "email": EMAIL
}

response = requests.get(SEARCH_URL, params=params)
data = response.json()

pmc_ids = data["esearchresult"]["idlist"]

print("Found papers:", len(pmc_ids))

Found papers: 300


In [ ]:
FETCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

def download_article(pmc_id):

    file_path = f"{DATA_DIR}/{pmc_id}.xml"

    # Skip if already downloaded
    if os.path.exists(file_path):
        return

    params = {
        "db": "pmc",
        "id": pmc_id,
        "retmode": "xml",
        "email": EMAIL
    }

    try:
        r = requests.get(FETCH_URL, params=params, timeout=30)
        r.raise_for_status()

        with open(file_path, "wb") as f:
            f.write(r.content)

    except Exception as e:
        print(f"Failed: {pmc_id}", e)

    time.sleep(1)  # Respect NCBI rate limits

In [7]:
for pmc_id in tqdm(pmc_ids):
    download_article(pmc_id)

100%|██████████| 300/300 [05:01<00:00,  1.01s/it]


In [8]:
len(os.listdir(DATA_DIR))

300